# Шаг 3. Реранжирование cross-encoder'ом

Векторный поиск из шага 2 быстрый, но у него есть врождённое ограничение: запрос и статья
кодируются в векторы **независимо**, и модель не видит их вместе. **Cross-encoder** (я использую
`BAAI/bge-reranker-v2-m3`) устроен иначе: он получает на вход пару «запрос + фрагмент статьи»
одним текстом и читает их совместно, слово за словом. Это на порядки дороже - поэтому им нельзя
прогнать все 793 статьи на каждый запрос, - но существенно точнее. Отсюда классическая
двухступенчатая схема: дешёвый поиск отбирает топ-50 кандидатов, дорогой cross-encoder
их перечитывает и переоценивает (**reranking** - повторное ранжирование).

Каждому кандидату cross-encoder читает не всю статью, а **два чанка, ближайших к запросу
по векторной близости** (они выбраны в шаге 2): релевантные места уже найдены, читать
остальное - трата GPU-времени без пользы для качества.

Модель применяется **zero-shot** - как есть, без дообучения на моих данных: она обучена
на большом многоязычном корпусе пар «запрос-документ» и русский понимает из коробки.

Запуск: Colab с GPU (**T4**). Скоринг всех пар (~100 тысяч) с нуля занимает 40-60 минут;
результат кэшируется на Диск, повторный запуск - меньше минуты.

## 0. Установка зависимостей

In [1]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--upgrade-strategy", "only-if-needed",
        "transformers>=4.50,<5", "accelerate>=1,<2", "huggingface-hub>=0.28,<1",
        "pyarrow", "tqdm",
    ])
    print("Зависимости установлены.")
else:
    print("Не Colab: зависимости из requirements.txt.")

Зависимости установлены.


## 1. Google Диск и кандидаты из шага 2

In [2]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/avito_rag")
else:
    PROJECT_ROOT = Path.cwd().resolve().parent / "avito_rag"

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CANDIDATES_DIR = ARTIFACTS_DIR / "candidates"
RERANKER_DIR = ARTIFACTS_DIR / "reranker"
RERANKER_DIR.mkdir(parents=True, exist_ok=True)

required = {
    "development": CANDIDATES_DIR / "pairs_development_fusion_top_50.parquet",
    "holdout": CANDIDATES_DIR / "pairs_holdout_fusion_top_50.parquet",
    "test": CANDIDATES_DIR / "pairs_test_fusion_top_50.parquet",
}
missing = [str(p) for p in required.values() if not p.is_file()]
if missing:
    raise FileNotFoundError("Сначала выполните ноутбук 02 — не найдены файлы:\n" + "\n".join(missing))
print("PROJECT_ROOT:", PROJECT_ROOT)

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/avito_rag


In [3]:
from __future__ import annotations

import gc
import hashlib
import json
import re
import time
from typing import Any, Iterable

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

RANDOM_STATE = 42
TOP_K = 10
CANDIDATE_N = 50
RERANKER_NAME = "BAAI/bge-reranker-v2-m3"
RERANK_MAX_LENGTH = 512   # максимум токенов пары «запрос + чанк»

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RERANKER_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
RERANK_BATCH_SIZE = 8 if DEVICE == "cuda" else 2

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

pd.set_option("display.max_colwidth", 120)

print("Device:", DEVICE)
if DEVICE != "cuda":
    print("Внимание: без GPU скоринг займёт много часов.")

pairs_by_split = {name: pd.read_parquet(path) for name, path in required.items()}
for name, pairs in pairs_by_split.items():
    print(f"{name}: {len(pairs):,} пар, {pairs['query_id'].nunique()} запросов")


def stable_hash(values: Iterable[Any]) -> str:
    digest = hashlib.sha256()
    for value in values:
        digest.update(str(value).encode("utf-8"))
        digest.update(b"\x00")
    return digest.hexdigest()


def pairs_hash(pairs: pd.DataFrame) -> str:
    columns = ["query_id", "article_id", "chunk_index", "query_text", "chunk_text"]
    values = pairs[columns].astype(str).agg("\x1f".join, axis=1).tolist()
    return stable_hash(values)

Device: cuda
development: 37,390 пар, 400 запросов
holdout: 9,311 пар, 100 запросов
test: 46,542 пар, 500 запросов


## 2. Скоринг пар с кэшированием

Функция ниже прогоняет cross-encoder по всем парам одного сплита и выдаёт «логит» -
число, монотонно отражающее уверенность модели в релевантности (чем больше, тем релевантнее;
в вероятность не перевожу, для ранжирования важен только порядок).

Инженерные детали, которые считаю важными для воспроизводимости:

- результат кэшируется на Диск вместе с метаданными, включающими SHA-256 всех пар -
  кэш переиспользуется только если входные данные буквально совпадают;
- ревизия (конкретный коммит весов) модели фиксируется при первом запуске и переиспользуется
  при последующих - защита от тихого обновления модели на Hugging Face;
- при нехватке GPU-памяти размер батча автоматически уменьшается вдвое, а не роняет ноутбук.

In [4]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Фиксация ревизии: если метаданные от прошлых запусков есть, используется та же ревизия весов.
fixed_reranker_revision = None
for metadata_path in sorted(RERANKER_DIR.glob("bge_reranker_v2_m3_*_scores.metadata.json")):
    fixed_reranker_revision = json.loads(metadata_path.read_text(encoding="utf-8")).get("resolved_revision")
    if fixed_reranker_revision:
        break
print("Зафиксированная ревизия reranker:", fixed_reranker_revision)

_reranker = {"model": None, "tokenizer": None}


def load_reranker():
    if _reranker["model"] is None:
        kwargs = {}
        if fixed_reranker_revision:
            kwargs["revision"] = fixed_reranker_revision
        _reranker["tokenizer"] = AutoTokenizer.from_pretrained(RERANKER_NAME, **kwargs)
        _reranker["model"] = AutoModelForSequenceClassification.from_pretrained(
            RERANKER_NAME, torch_dtype=RERANKER_DTYPE, low_cpu_mem_usage=True, **kwargs
        )
        _reranker["model"].to(DEVICE)
        _reranker["model"].eval()
    return _reranker["model"], _reranker["tokenizer"]


def score_split_with_cache(split_name: str, pairs: pd.DataFrame) -> pd.DataFrame:
    cache_path = RERANKER_DIR / f"bge_reranker_v2_m3_{split_name}_fusion_top_50_scores.parquet"
    metadata_path = cache_path.with_suffix(".metadata.json")

    expected_metadata = {
        "model_name": RERANKER_NAME,
        "resolved_revision": fixed_reranker_revision,
        "candidate_strategy": "fusion_top_50",
        "max_length": RERANK_MAX_LENGTH,
        "pair_count": len(pairs),
        "pairs_sha256": pairs_hash(pairs),
    }

    if cache_path.is_file() and metadata_path.is_file():
        cached_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        cached_pairs = pd.read_parquet(cache_path)
        key_columns = ["query_id", "article_id", "chunk_index"]
        cache_valid = all(cached_metadata.get(k) == v for k, v in expected_metadata.items() if k != "resolved_revision")
        cache_valid = cache_valid and len(cached_pairs) == len(pairs)
        cache_valid = cache_valid and "reranker_logit" in cached_pairs.columns
        cache_valid = cache_valid and cached_pairs[key_columns].reset_index(drop=True).equals(
            pairs[key_columns].reset_index(drop=True)
        )
        if cache_valid:
            print(f"[{split_name}] кэш найден: {cache_path.name}")
            return cached_pairs
        print(f"[{split_name}] кэш устарел, пересчитываю.")

    model, tokenizer = load_reranker()
    queries = pairs["query_text"].astype(str).tolist()
    passages = pairs["chunk_text"].astype(str).tolist()
    logits = np.empty(len(pairs), dtype=np.float32)

    batch_size = RERANK_BATCH_SIZE
    cursor = 0
    started = time.perf_counter()
    progress = tqdm(total=len(pairs), desc=f"Cross-encoder ({split_name})")

    while cursor < len(pairs):
        current = min(batch_size, len(pairs) - cursor)
        try:
            encoded = tokenizer(
                queries[cursor:cursor + current], passages[cursor:cursor + current],
                padding=True, truncation=True, max_length=RERANK_MAX_LENGTH, return_tensors="pt",
            )
            encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
            with torch.inference_mode():
                output = model(**encoded, return_dict=True)
                logits[cursor:cursor + current] = output.logits.reshape(-1).float().cpu().numpy()
            cursor += current
            progress.update(current)
            del encoded, output
        except torch.cuda.OutOfMemoryError:
            if DEVICE != "cuda" or batch_size == 1:
                progress.close()
                raise
            batch_size = max(1, batch_size // 2)
            torch.cuda.empty_cache()
            print(f"\nCUDA OOM: батч уменьшен до {batch_size}.")

    progress.close()
    seconds = time.perf_counter() - started

    scored = pairs.copy()
    scored["reranker_logit"] = logits

    actual_revision = getattr(model.config, "_commit_hash", None) or fixed_reranker_revision
    metadata = {
        **expected_metadata,
        "resolved_revision": actual_revision,
        "dtype": str(RERANKER_DTYPE), "device": DEVICE,
        "seconds": seconds, "pairs_per_second": len(pairs) / seconds if seconds else None,
        "final_batch_size": batch_size,
        "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    }
    scored.to_parquet(cache_path, index=False)
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[{split_name}] {len(pairs):,} пар за {seconds:.0f} с -> {cache_path.name}")
    return scored


scored_by_split = {}
for split_name, pairs in pairs_by_split.items():
    scored_by_split[split_name] = score_split_with_cache(split_name, pairs)

if _reranker["model"] is not None:
    _reranker["model"] = None
    _reranker["tokenizer"] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Зафиксированная ревизия reranker: 953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e
[development] кэш устарел, пересчитываю.


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

...

[development] 37,390 пар за 706 с -> bge_reranker_v2_m3_development_fusion_top_50_scores.parquet
[holdout] кэш устарел, пересчитываю.


Cross-encoder (holdout):   0%|          | 0/9311 [00:00<?, ?it/s]

[holdout] 9,311 пар за 178 с -> bge_reranker_v2_m3_holdout_fusion_top_50_scores.parquet
[test] кэш устарел, пересчитываю.


Cross-encoder (test):   0%|          | 0/46542 [00:00<?, ?it/s]

[test] 46,542 пар за 889 с -> bge_reranker_v2_m3_test_fusion_top_50_scores.parquet


## 3. Насколько реранкер улучшил поиск

Проверяю вклад этапа честным способом - метриками на размеченных данных. Скоры двух чанков
кандидата агрегирую в скор статьи (среднее), нормализую per-query и смешиваю с retrieval-скором:
`0.75 × reranker + 0.25 × fusion`. Небольшая примесь fusion - страховка: если cross-encoder
по двум чанкам «не понял» статью, сигнал первой ступени не даёт ей провалиться.

Сравниваю с fusion-бейзлайном из шага 2 на development и один раз на holdout.

In [5]:
def unique_top_k(prediction, k: int) -> list[int]:
    result, seen = [], set()
    for value in prediction:
        article_id = int(value)
        if article_id in seen:
            continue
        seen.add(article_id)
        result.append(article_id)
        if len(result) >= k:
            break
    return result


def average_precision_at_k(ground_truth, prediction, k: int = TOP_K) -> float:
    relevant = set(int(x) for x in ground_truth)
    if not relevant:
        return 0.0
    hits, precision_sum = 0, 0.0
    for rank, article_id in enumerate(unique_top_k(prediction, k), start=1):
        if article_id in relevant:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / min(len(relevant), k)


def evaluate_rankings(query_table: pd.DataFrame, rankings: list[list[int]]) -> dict[str, float]:
    truths = query_table["ground_truth"].tolist()
    map10 = float(np.mean([average_precision_at_k(gt, p) for gt, p in zip(truths, rankings)]))
    recall = float(np.mean([
        len(set(int(x) for x in gt) & set(p[:TOP_K])) / len(set(int(x) for x in gt))
        for gt, p in zip(truths, rankings)
    ]))
    hit = float(np.mean([bool(set(int(x) for x in gt) & set(p[:TOP_K])) for gt, p in zip(truths, rankings)]))
    return {"MAP@10": map10, "Recall@10": recall, "HitRate@10": hit}


def normalize_ground_truth(value) -> list[int]:
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        return [int(x) for x in value]
    return [int(x) for x in re.findall(r"\d+", str(value))]


def minmax_by_query(frame: pd.DataFrame, column: str) -> pd.Series:
    minimum = frame.groupby("query_id")[column].transform("min")
    shifted = frame[column] - minimum
    maximum = shifted.groupby(frame["query_id"]).transform("max")
    return shifted.div(maximum.where(maximum > 0, 1.0))


def rankings_from_scores(candidate_frame: pd.DataFrame, score_column: str, query_order) -> list[list[int]]:
    ranking_map = {}
    for query_id, group in candidate_frame.groupby("query_id", sort=False):
        ordered = group.sort_values([score_column, "article_id"], ascending=[False, True])
        ranking_map[int(query_id)] = unique_top_k(ordered["article_id"].astype(int).tolist(), TOP_K)
    return [ranking_map[int(query_id)] for query_id in query_order]


report_rows = []
for split_name in ("development", "holdout"):
    scored = scored_by_split[split_name].copy()
    scored["ground_truth"] = scored["ground_truth"].map(normalize_ground_truth)

    # Агрегация: два чанка -> одна строка на (запрос, статья).
    candidates = (
        scored.sort_values(["query_id", "article_id", "chunk_dense_rank"])
        .groupby(["query_id", "article_id"], sort=False, as_index=False)
        .agg(
            fusion_score=("fusion_score", "first"),
            reranker_top2_mean=("reranker_logit", "mean"),
            ground_truth=("ground_truth", "first"),
            query_text=("query_text", "first"),
        )
    )
    candidates["reranker_norm"] = minmax_by_query(candidates, "reranker_top2_mean")
    candidates["retrieval_norm"] = minmax_by_query(candidates, "fusion_score")
    candidates["reranked_score"] = 0.75 * candidates["reranker_norm"] + 0.25 * candidates["retrieval_norm"]

    query_table = candidates[["query_id", "ground_truth"]].drop_duplicates("query_id").sort_values("query_id").reset_index(drop=True)
    query_order = query_table["query_id"].tolist()

    for method, column in [("fusion (шаг 2)", "fusion_score"), ("reranked (шаг 3)", "reranked_score")]:
        metrics = evaluate_rankings(query_table, rankings_from_scores(candidates, column, query_order))
        report_rows.append({"split": split_name, "method": method, **metrics})

report = pd.DataFrame(report_rows)
display(report.round(4))

,split,method,MAP@10,Recall@10,HitRate@10
0,development,fusion (шаг 2),0.4911,0.8306,0.89
1,development,reranked (шаг 3),0.5501,0.8973,0.94
2,holdout,fusion (шаг 2),0.5711,0.9017,0.92
3,holdout,reranked (шаг 3),0.6480,0.9450,0.96


## Выводы

1. Cross-encoder ощутимо поднимает качество поверх fusion (на development примерно
   +0.05 MAP@10, на holdout ещё больше) - совместное чтение запроса и текста действительно
   ловит то, что независимые векторы не видят.
2. Скоры всех пар для development, holdout и test сохранены на Диск с контрольными хэшами -
   шаг 4 работает только с ними и не требует GPU.
3. Важно, что здесь модель ещё ничему не училась на моих данных: это zero-shot оценка,
   поэтому её прирост гарантированно переносится на test. Выжать больше из имеющихся
   50 размеченных на запрос кандидатов пробует следующий шаг - обучаемое ранжирование.